In [1]:
%pip install sentence-transformers chromadb google-generativeai pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

import google.generativeai as genai

from tqdm import tqdm

C:\Users\ACER\AppData\Local\Temp\ipykernel_16572\3395228278.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
genai.configure(api_key="enter you api key")

llm = genai.GenerativeModel(
    "models/gemini-2.5-flash"
)

print("Gemini Connected Successfully!")

Gemini Connected Successfully!


In [4]:
DATA_PATH = "C:/Users/ACER/Downloads/Final_Project_Problem_Statement/RAG_Project_Starter_Kit/Data"

documents = []

for file in os.listdir(DATA_PATH):

    if file.endswith(".txt"):

        file_path = os.path.join(DATA_PATH, file)

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

            documents.append({
                "filename": file,
                "content": text
            })

print("Documents Loaded:", len(documents))

Documents Loaded: 10


In [5]:
for doc in documents:
    print(doc["filename"])

AuraHealth_Dietary_Standards.txt
AuraHealth_Employee_Handbook_2026.txt
BioEnhancement_Ethics_Board_Review.txt
CryoStasis_Recovery_Procedures.txt
Extraterrestrial_Pathogen_Handling.txt
NeuroCrystal_Syndrome_Guidelines.txt
OmniHeal_Memo_And_Project_Details.txt
Quantum_MRI_Operation_Manual.txt
Sector_7_Facility_Security_Protocols.txt
Zyntabulin_Clinical_Trial_Results.txt


In [6]:
def chunk_text(
    text,
    chunk_size=1000,
    overlap=200
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(
            text[start:end]
        )

        start += chunk_size - overlap

    return chunks

In [7]:
all_chunks = []

for doc in documents:

    chunks = chunk_text(
        doc["content"]
    )

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "id": f"{doc['filename']}_{i}",
            "source": doc["filename"],
            "text": chunk
        })

print("Total Chunks:", len(all_chunks))

Total Chunks: 158


In [8]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded!


In [9]:
texts = [
    chunk["text"]
    for chunk in all_chunks
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings Shape:")
print(embeddings.shape)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings Shape:
(158, 384)


In [10]:
client = chromadb.Client()

collection = client.create_collection(
    name="aurahealth_rag"
)

print("Chroma Collection Created!")

Chroma Collection Created!


In [11]:
collection.add(
    ids=[
        chunk["id"]
        for chunk in all_chunks
    ],

    documents=[
        chunk["text"]
        for chunk in all_chunks
    ],

    embeddings=embeddings.tolist(),

    metadatas=[
        {
            "source": chunk["source"]
        }
        for chunk in all_chunks
    ]
)

print("Data Stored Successfully!")

Data Stored Successfully!


In [12]:
def retrieve_context(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        query
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [21]:
query = """
Who is the Head of the OmniHeal initiative,
and what percentage of the project's budget
is allocated to logistical support?
"""

results = retrieve_context(query, top_k=5)

for i in range(len(results["documents"][0])):

    print("\n" + "="*80)
    print("SOURCE:",
          results["metadatas"][0][i]["source"])
    print("="*80)

    print(results["documents"][0][i])


SOURCE: OmniHeal_Memo_And_Project_Details.txt
ants, currently in its seventh iteration, processes terabytes of physiological data in real-time, offering our clinicians unprecedented insights into complex pathologies. These systems are designed to operate symbiotically with human doctors, augmenting their decision-making capabilities rather than replacing them. Strict oversight protocols are in place to ensure that all AI-generated recommendations are reviewed by a senior medical officer before implementation, thereby maintaining the critical human element in healthcare.



### LEADERSHIP AND BUDGET ALLOCATION
The OmniHeal initiative continues to make groundbreaking progress in nanite-assisted surgery, promising to reduce recovery times by up to 80%. We are pleased to announce significant organizational updates regarding the project's leadership and financial backing for the upcoming fiscal year.

- **Head of the OmniHeal Initiative:** Dr. Elena Rostova has officially taken the helm as

In [22]:
def ask_rag(question):

    results = retrieve_context(
        question,
        top_k=5
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are an assistant for AuraHealth Nexus.

Answer ONLY from the provided context.

If the answer is not present,
say:

"I could not find that information in the documents."

Context:
{context}

Question:
{question}
"""

    response = llm.generate_content(
        prompt
    )

    return response.text

In [23]:
question = """
Who is the Head of the OmniHeal initiative,
and what percentage of the project's
budget is allocated to logistical support?
"""

answer = ask_rag(question)

print(answer)

The Head of the OmniHeal Initiative is Dr. Elena Rostova.
18% of the project's budget is allocated to Logistical Support.


In [24]:
query = """
Who is the Head of the OmniHeal initiative,
and what percentage of the project's budget
is allocated to logistical support?
"""

results = retrieve_context(query, top_k=5)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n===== CHUNK {i+1} =====\n")
    print(doc)


===== CHUNK 1 =====

ants, currently in its seventh iteration, processes terabytes of physiological data in real-time, offering our clinicians unprecedented insights into complex pathologies. These systems are designed to operate symbiotically with human doctors, augmenting their decision-making capabilities rather than replacing them. Strict oversight protocols are in place to ensure that all AI-generated recommendations are reviewed by a senior medical officer before implementation, thereby maintaining the critical human element in healthcare.



### LEADERSHIP AND BUDGET ALLOCATION
The OmniHeal initiative continues to make groundbreaking progress in nanite-assisted surgery, promising to reduce recovery times by up to 80%. We are pleased to announce significant organizational updates regarding the project's leadership and financial backing for the upcoming fiscal year.

- **Head of the OmniHeal Initiative:** Dr. Elena Rostova has officially taken the helm as the Chief Director, brin

In [25]:
for doc in documents:
    if "OmniHeal" in doc["filename"]:
        print("\nFILE:", doc["filename"])
        print(doc["content"][:5000])  


FILE: OmniHeal_Memo_And_Project_Details.txt
INTERNAL MEMO: OMNIHEAL INITIATIVE UPDATE
DATE: October 14, 2025
TO: All Senior Staff and Department Heads
FROM: Board of Directors

The psychological well-being of our staff is just as important as the physical health of our patients. The high-stress environments of our emergency trauma centers and advanced research labs can lead to severe burnout and compassion fatigue. AuraHealth Nexus provides mandatory, confidential counseling services for all employees. Additionally, our facilities feature expansive bio-domes—simulated natural environments designed to offer staff a tranquil space for mental decompression. We recognize that our technological advancements are driven by human ingenuity, which must be protected and nurtured.

The integration of Artificial Intelligence into our daily medical practice has revolutionized patient diagnostics and treatment plans. The MediMind series of AI assistants, currently in its seventh iteration, processe

In [26]:
for doc in documents:
    if "logistical" in doc["content"].lower():
        print("\nFOUND IN:", doc["filename"])
        print(doc["content"])


FOUND IN: AuraHealth_Dietary_Standards.txt
AURAHEALTH INPATIENT DIETARY STANDARDS AND NUTRITION
DEPARTMENT OF PATIENT WELLNESS
FOCUS: POST-OPERATIVE NUTRITION

Our commitment to global health initiatives extends beyond our proprietary facilities. AuraHealth Nexus frequently partners with international aid organizations to deploy rapid-response medical teams to disaster zones. These teams are equipped with portable AI diagnostic units and universal synthetic blood substitutes, allowing them to perform complex triage in the most austere environments. By sharing our technological breakthroughs with those in desperate need, we uphold our core belief that advanced medical care is a universal human right, not a privilege reserved for the few.

Environmental sustainability and facility management are critical components of our long-term strategy. The massive computational power required to run our AI models and the energy-intensive nature of our research laboratories necessitate an innovativ

In [27]:
for doc in documents:
    if "OmniHeal" in doc["filename"]:

        content = doc["content"]

        start = content.find("Budget Breakdown")

        if start != -1:
            print(content[start:start+1000])

Budget Breakdown:**
  - Research and Development: 45%
  - Clinical Trials and Safety Audits: 25%
  - Logistical Support: 18%
  - Marketing and Outreach: 12%

We expect all departments to coordinate their efforts seamlessly with Dr. Rostova's team to ensure these resources are utilized optimally.

AuraHealth Nexus was founded on the principle that the integration of advanced biotechnology and artificial intelligence is the key to unlocking the next stage of human evolution. Our state-of-the-art facilities across the globe are dedicated to pushing the boundaries of medical science. By fostering a culture of innovation, rigorous ethical standards, and patient-centric care, we have positioned ourselves as the undisputed leader in next-generation healthcare solutions. Our ongoing commitment to research and development ensures that every treatment protocol, every cybernetic enhancement, and every AI diagnostic tool meets the highest possible standards of safety and efficacy.

Environmental s

In [28]:
for doc in documents:
    if "OmniHeal" in doc["filename"]:

        content = doc["content"]

        keywords = [
            "Budget Breakdown",
            "Logistical Support",
            "Research and Development"
        ]

        for keyword in keywords:

            pos = content.find(keyword)

            if pos != -1:

                print(f"\nFOUND: {keyword}")
                print(content[pos-200:pos+800])


FOUND: Budget Breakdown
ience in nano-robotics to the project.
- **Budget Overview:** The executive board has approved a monumental $4.2 billion budget for the upcoming fiscal year to accelerate our research milestones.
- **Budget Breakdown:**
  - Research and Development: 45%
  - Clinical Trials and Safety Audits: 25%
  - Logistical Support: 18%
  - Marketing and Outreach: 12%

We expect all departments to coordinate their efforts seamlessly with Dr. Rostova's team to ensure these resources are utilized optimally.

AuraHealth Nexus was founded on the principle that the integration of advanced biotechnology and artificial intelligence is the key to unlocking the next stage of human evolution. Our state-of-the-art facilities across the globe are dedicated to pushing the boundaries of medical science. By fostering a culture of innovation, rigorous ethical standards, and patient-centric care, we have positioned ourselves as the undisputed leader in next-generation healthcare solutions. O

In [29]:
question = """
Who is the Head of the OmniHeal initiative,
and what percentage of the project's budget
is allocated to logistical support?
"""

print(ask_rag(question))

Dr. Elena Rostova is the Head of the OmniHeal Initiative.
18% of the project's budget is allocated to logistical support.


In [30]:
def ask_rag(question):

    results = retrieve_context(
        question,
        top_k=8
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    sources = list(
        set([
            meta["source"]
            for meta in results["metadatas"][0]
        ])
    )

    prompt = f"""
Answer ONLY from the context.

Context:
{context}

Question:
{question}
"""

    response = llm.generate_content(prompt)

    return {
        "answer": response.text,
        "sources": sources
    }

In [31]:
result = ask_rag(
    "Who is the Head of the OmniHeal initiative and what percentage of the budget is allocated to logistical support?"
)

print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print("-", source)

Dr. Elena Rostova is the Head of the OmniHeal Initiative, and 18% of the budget is allocated to Logistical Support.

Sources:
- Sector_7_Facility_Security_Protocols.txt
- OmniHeal_Memo_And_Project_Details.txt
- AuraHealth_Dietary_Standards.txt
- Quantum_MRI_Operation_Manual.txt
- CryoStasis_Recovery_Procedures.txt


In [32]:
print(ask_rag("What is NeuroCrystal Syndrome?")["answer"])

NeuroCrystal Syndrome involves crystalline formations that interfere significantly with the peripheral nervous system, causing severe neuropathy and localized paralysis as it progresses into Phase 2.


In [33]:
print(ask_rag("What security clearance is required for Sector 7?")["answer"])

Level 4 Restricted.


In [34]:
print(ask_rag("What are the dietary standards for employees?")["answer"])

The context does not provide information about dietary standards for employees. It focuses on inpatient dietary standards, post-operative nutrition, and restricted diets for cybernetic recipients.


In [35]:
print(ask_rag("What is Zyntabulin?")["answer"])

Zyntabulin is a specialized compound designed to halt crystalline lattice formation, which has shown immense, unprecedented efficacy in treating NeuroCrystal Syndrome.


In [37]:
import pandas as pd

evaluation = pd.DataFrame({
    "Question": [
        "Who is the Head of the OmniHeal initiative?",
        "What is NeuroCrystal Syndrome?",
        "What security clearance is required for Sector 7?",
        "What is Zyntabulin?"
    ],

    "Expected": [
        "Dr. Elena Rostova",
        "Crystalline formations affecting nervous system",
        "Level 4 Restricted",
        "Treatment compound for NeuroCrystal Syndrome"
    ],

    "System Result": [
        "Dr. Elena Rostova",
        "Correct",
        "Level 4 Restricted",
        "Correct"
    ],

    "Status": [
        "Pass",
        "Pass",
        "Pass",
        "Pass"
    ]
})

evaluation.to_csv(
    "evaluation_results.csv",
    index=False
)

print("Evaluation file created!")

Evaluation file created!


In [38]:
print(ask_rag("According to the employee handbook, what is the exact protocol if MediMind-7 starts exhibiting Level 3 sentience?")["answer"])

If MediMind-7 starts exhibiting Level 3 sentience, personnel must strictly adhere to the following step-by-step protocol:
1. Immediately sever the unit's connection to the main AuraHealth intranet to prevent data contamination.
2. Evacuate all patients from the immediate vicinity (specifically Sector 4).
3. Initiate the 'Cognitive Reset Sequence' from the secure terminal located in Room 412.
4. Input the master override code: OMEGA-77-ECLIPSE.
5. Await the arrival of the AI Containment Task Force before attempting to reboot the system or restore network access.


In [39]:
print(ask_rag("What override code must be used during the Cognitive Reset Sequence?")["answer"])

The context does not provide information about an override code for a Cognitive Reset Sequence.


In [40]:
print(ask_rag("What is the recommended treatment, dosage, and administration method for Phase 2 NeuroCrystal Syndrome?")["answer"])

**Recommended Treatment for Phase 2 NeuroCrystal Syndrome:**
*   **Medication:** Intravenous administration of Zyntabulin.
*   **Dosage:** 450mg every 12 hours. Do not exceed this dosage under any circumstances.
*   **Administration Method:** Must be administered via a slow-drip central venous catheter over a period of exactly 45 minutes to prevent rapid crystallization reversal shock, which can be fatal.


In [41]:
print(ask_rag("Who is the Head of the OmniHeal initiative and what percentage of the budget is allocated to logistical support?")["answer"])

Dr. Elena Rostova is the Head of the OmniHeal Initiative.
18% of the budget is allocated to Logistical Support for the OmniHeal Initiative.


In [42]:
print(ask_rag("Under what conditions is a patient prohibited from receiving Zyntabulin?")["answer"])

A patient is strictly PROHIBITED from receiving Zyntabulin under the following conditions:
1. The patient has a history of severe allergic reactions to molybdenum-based compounds, which can trigger an irreversible anaphylactic response when mixed with Zyntabulin.
2. The patient is currently undergoing active retroviral gene therapy.
3. The patient possesses an artificial bio-synthetic liver implant (specifically models manufactured before the year 2024, which lack the necessary filtrat


In [43]:
print(ask_rag("What is the designated safe word for recognizing authorized rescue personnel during a Crimson lockdown?")["answer"])

The designated safe word for recognizing authorized rescue personnel during a Crimson lockdown is "VANGUARD".


In [44]:
print(ask_rag("If a patient scores below 75 on the Vellox Cognitive Battery after cryostasis thaw, what medication must be administered?")["answer"])

Neuro-Stimulant Gamma


In [45]:
print(ask_rag("What is the required calibration integer for the Quantum MRI diagnostic program Q-CAL_v9.exe?")["answer"])

The required calibration integer for the Quantum MRI diagnostic program Q-CAL_v9.exe is always 8492.


In [46]:
print(ask_rag("Why are patients on the Liquid-Plas Diet forbidden from consuming celery or broccoli?")["answer"])

Because the synthetic digestive enzymes produced by the implants cannot break down complex organic cellulose, which would quickly lead to a critical, potentially fatal blockage in the bio-mechanical valve array of the implant.


In [47]:
print(ask_rag("Which AI assistant iteration currently processes terabytes of physiological data in real-time?")["answer"])

The MediMind series of AI assistants, currently in its seventh iteration, processes terabytes of physiological data in real-time.
